# Amaliy mashgʻulot 3. Morfologik tahlil va lemmatizatsiya

3-laboratoriyada stemmer va lemmatizatorni solishtirdingiz, koʻp maʼnoli tahlillarni koʻrdingiz va soʻzni chekli avtomat orqali oʻtkazdingiz. Bu yerda xuddi shuni kodda bajaramiz:

1. ot shakllarini qoidalar bilan yasaymiz: koʻplik, egalik, kelishik, allomorflar (-ga/-ka/-qa) va tovush almashinuvi (k → g, q → gʻ);
2. qoʻshimchalarni oʻngdan chapga kesadigan stemmer yozamiz va uning xatolarini koʻramiz;
3. kichik lugʻat va chekli avtomat asosida barcha tahlillarni qaytaradigan morfologik tahlilchi yozamiz;
4. matnda stemming va lemmatizatsiyani solishtiramiz.

Oʻzbek tili uchun tayyor vositalar kam: rus tilidagi `pymorphy3` kabi keng qoʻllanadigan, lugʻatga asoslangan tahlilchi yoʻq, `nltk` dagi Snowball stemmerlari orasida ham oʻzbek tili yoʻq. Shuning uchun hamma narsani oʻzimiz yozamiz.

*«NLP algoritmlari» kursining 3-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

## 1. Shakl yasash: qoʻshimchalar tartibi va allomorflar

Oʻzbek otida qoʻshimchalar qatʼiy tartibda keladi: **oʻzak + koʻplik + egalik + kelishik** (kitob-lar-imiz-dan). Bir qoʻshimchaning bir nechta shakli (**allomorf**) bor:

- egalik: undoshdan keyin -im, -ing, -i, -imiz, -ingiz; unlidan keyin -m, -ng, -si, -miz, -ngiz;
- joʻnalish kelishigi: -ga, lekin k dan keyin -ka, q dan keyin -qa (terakka, qishloqqa);
- koʻp boʻgʻinli soʻzlarda unli bilan boshlanadigan egalik qoʻshimchasi oldidan k → g, q → gʻ (terak → teragi, qishloq → qishlogʻi);
- baʼzi soʻzlarda unli tushadi: ogʻiz → ogʻzi, shahar → shahri.

In [ ]:
import re
import pandas as pd
pd.set_option("display.width", 200)

def ends_vowel(s):
    return s.endswith(("a", "e", "i", "o", "u", "oʻ"))

def syllables(s):
    return len(re.findall(r"[aeiou]", s))

ALTERN = {"k": "g", "q": "gʻ"}                                           # terak → teragi
VOWEL_LOSS = {"ogʻiz": "ogʻz", "shahar": "shahr", "oʻgʻil": "oʻgʻl"}     # ogʻiz → ogʻzi
POSS = [("I sh. birlik", "im", "m"), ("II sh. birlik", "ing", "ng"), ("III shaxs", "i", "si"),
        ("I sh. koʻplik", "imiz", "miz"), ("II sh. koʻplik", "ingiz", "ngiz")]
CASES = [("bosh", ""), ("qaratqich", "ning"), ("tushum", "ni"), ("joʻnalish", "ga"),
         ("oʻrin-payt", "da"), ("chiqish", "dan")]

def add_possessive(stem, after_consonant, after_vowel):
    if ends_vowel(stem):
        return stem + after_vowel
    if stem in VOWEL_LOSS:
        stem = VOWEL_LOSS[stem]
    elif stem[-1] in ALTERN and syllables(stem) > 1:
        stem = stem[:-1] + ALTERN[stem[-1]]
    return stem + after_consonant

def add_case(stem, case):
    if case == "ga" and stem.endswith("k"):
        return stem + "ka"
    if case == "ga" and stem.endswith("q"):
        return stem + "qa"
    return stem + case

def noun_form(root, plural=False, poss=None, case=""):
    w = root + ("lar" if plural else "")
    if poss:
        w = add_possessive(w, poss[1], poss[2])
    return add_case(w, case)

table = pd.DataFrame([[noun_form("qishloq", False, p, c) for _, c in CASES] for p in [None] + POSS],
                     index=["egaliksiz"] + [p[0] for p in POSS], columns=[n for n, _ in CASES])
print(table)
print()
for root in ["kitob", "bola", "terak", "yuk", "ogʻiz", "shahar"]:
    print(f"{root:<7} III sh.: {noun_form(root, poss=POSS[2]):<9} joʻnalish: {noun_form(root, case='ga'):<9}"
          f" koʻplik+I sh.koʻp.+chiqish: {noun_form(root, True, POSS[3], 'dan')}")

Bitta otdan nechta shakl chiqadi? Koʻplik (2) × egalik (6) × kelishik (6) = 72 ta. Rus otida esa 12 ga yaqin (6 kelishik × 2 son). Shu sababli oʻzbek matnida turli soʻz shakllari koʻp, har bir shakl esa kam uchraydi: stemming va lemmatizatsiya oʻzbek tili uchun ayniqsa muhim.

In [ ]:
forms = {noun_form("kitob", pl, ps, cs) for pl in (False, True) for ps in [None] + POSS for _, cs in CASES}
print(len(forms), "ta shakl, masalan:", ", ".join(sorted(forms, key=len)[-5:]))

## 2. Qoʻshimchalarni kesuvchi stemmer

Stemmer lugʻatsiz ishlaydi: soʻz oxiridan qoʻshimchalarni **oʻngdan chapga**, morfotaktik tartibga teskari holda kesadi: avval kelishik, keyin egalik, keyin koʻplik. Feʼl qoʻshimchalari alohida roʻyxatda. Allomorflarni hisobga olamiz (-ka faqat k dan keyin, -si faqat unlidan keyin), unli bilan boshlanuvchi egalik kesilgandan keyin esa g → k, gʻ → q **tiklanadi**. Oʻzak kamida 2 harf qolishi kerak. Har bir kesish chop etiladi.

In [ ]:
CASE_SUF = ["ning", "dan", "ni", "ga", "ka", "qa", "da"]
POSS_SUF = [("ingiz", "C"), ("ngiz", "V"), ("imiz", "C"), ("miz", "V"), ("ing", "C"), ("ng", "V"),
            ("im", "C"), ("m", "V"), ("si", "V"), ("i", "C")]            # C: undoshdan keyin, V: unlidan keyin
NEG_VERB_SUF = ["mayapti", "maydi", "magan", "madim", "madi"]           # inkor -ma- + zamon
VERB_SUF = ["yapti", "aman", "adi", "ydi", "dim", "dik", "di", "gan"]
MIN_STEM = 2

def stem_uz(word, trace=False, restore=True):
    w, log = word.lower(), []
    def cut(suffix, label):
        nonlocal w
        w = w[: -len(suffix)]
        log.append((label, suffix))
    for suffix in NEG_VERB_SUF + VERB_SUF:                               # feʼl: eng uzunidan boshlab
        if w.endswith(suffix) and len(w) - len(suffix) >= MIN_STEM:
            cut(suffix, "feʼl")
            return (w, log) if trace else w
    for suffix in CASE_SUF:                                              # 1) kelishik
        rest = w[: -len(suffix)]
        ok = {"ka": rest.endswith("k"), "qa": rest.endswith("q"), "ga": not rest.endswith(("k", "q"))}.get(suffix, True)
        if w.endswith(suffix) and len(rest) >= MIN_STEM and ok:
            cut(suffix, "kelishik")
            break
    vowel_initial = False
    for suffix, after in POSS_SUF:                                       # 2) egalik
        rest = w[: -len(suffix)]
        if w.endswith(suffix) and len(rest) >= MIN_STEM and ends_vowel(rest) == (after == "V"):
            cut(suffix, "egalik")
            vowel_initial = suffix.startswith("i")
            break
    if w.endswith("lar") and len(w) - 3 >= MIN_STEM:                     # 3) koʻplik
        cut("lar", "koʻplik")
        vowel_initial = False
    if restore and vowel_initial and syllables(w) > 1:                   # 4) g → k, gʻ → q
        if w.endswith("gʻ"):
            w = w[:-2] + "q"; log.append(("tiklash", "gʻ → q"))
        elif w.endswith("g"):
            w = w[:-1] + "k"; log.append(("tiklash", "g → k"))
    return (w, log) if trace else w

for word in ["kitoblarimizdan", "qishlogʻimizga", "teragi", "terakka", "barglari", "olmadi", "oʻqiyapti", "bolalarga"]:
    stem, log = stem_uz(word, trace=True)
    print(f"{word:<16} → {stem:<9} [{'; '.join(f'{label}: {s}' for label, s in log) or 'hech narsa kesilmadi'}]")

Lugʻatsiz stemmer xato qiladi. **Ortiqcha kesish** (over-stemming): «kuni» → «ku» (kun + i emas, ku + ni deb oʻylaydi), «odam» → «oda» (-m egalik deb oʻylaydi), «oʻynadi» → «oʻyn» (-di oʻrniga -adi kesildi). **Kam kesish** (under-stemming): «ogʻzi» → «ogʻz» (unli tushishini bilmaydi). Tiklash qoidasi ham adashadi: «pedagogi» → «pedagok», chunki bu soʻzdagi g asl harf. Tiklashsiz esa «teragi» → «terag» boʻlib qoladi.

In [ ]:
cases = [("kuni", "kun"), ("odam", "odam"), ("oʻynadi", "oʻyna"), ("ogʻzi", "ogʻiz"), ("pedagogi", "pedagog"),
         ("teragi", "terak"), ("olma", "olma"), ("olmadi", "ol")]
rows = [(w, gold, stem_uz(w), stem_uz(w, restore=False)) for w, gold in cases]
df = pd.DataFrame(rows, columns=["soʻz", "toʻgʻri oʻzak", "stemmer", "tiklashsiz"])
df["toʻgʻri?"] = (df["stemmer"] == df["toʻgʻri oʻzak"]).map({True: "ha", False: "yoʻq"})
print(df.to_string(index=False))

## 3. Morfologik tahlilchi: lugʻat + chekli avtomat

Tahlilchi stemmerdan ikki narsa bilan farq qiladi: **oʻzaklar lugʻati** (soʻz turkumi bilan) va **chekli avtomat** (FSA), u qoʻshimchalar tartibini tekshiradi. Holatlar: OT → KOʻPLIK → EGALIK → KELISHIK; FEʼL → (INKOR) → zamon shakli; FEʼL → HARAKAT NOMI, undan keyin ot kabi egalik va kelishik kelishi mumkin (oʻqishimga). Har bir oʻtish bitta qoʻshimcha va uning sharti (unli/undoshdan keyin). Yalang feʼl oʻzagi va inkor shakli buyruq maylidagi shakl hisoblanadi (yoz! olma!), bu yerda qoʻshimcha nol (Ø).

Tahlilchi soʻzning **barcha** yoʻllarini qaytaradi. Ehtimollar uchun har bir (oʻzak, turkum) juftligiga taxminiy chastota beramiz: bu oʻylab topilgan oʻyinchoq sonlar, haqiqiy tizimda ular belgilangan korpusdan sanaladi.

In [ ]:
LEX = {"kitob": "ot", "qishloq": "ot", "terak": "ot", "bogʻ": "ot", "olma": "ot", "bola": "ot", "uka": "ot",
       "ota": "ot", "kun": "ot", "uy": "ot", "shahar": "ot", "ogʻiz": "ot", "soya": "ot", "oʻqish": "ot",
       "yoz": "ot feʼl", "ot": "ot feʼl", "ol": "feʼl", "oʻqi": "feʼl", "kel": "feʼl", "ber": "feʼl", "ter": "feʼl",
       "boq": "feʼl", "boʻl": "feʼl", "oʻyna": "feʼl", "qayt": "feʼl",
       "men": "olmosh", "biz": "olmosh", "baland": "sifat", "katta": "sifat", "uzun": "sifat", "bor": "boshqa"}
FREQ = {("yoz", "ot"): 6, ("yoz", "feʼl"): 9, ("ot", "ot"): 12, ("ot", "feʼl"): 3,
        ("olma", "ot"): 10, ("ol", "feʼl"): 30, ("oʻqish", "ot"): 4, ("oʻqi", "feʼl"): 20}   # oʻyinchoq sonlar

after_vowel = ends_vowel
after_consonant = lambda s: not ends_vowel(s)
FSA = {s: [] for s in ["OT", "KOʻPLIK", "EGALIK", "KELISHIK", "FEʼL", "INKOR", "HARAKAT NOMI", "OXIRI"]}
FSA["OT"].append(("KOʻPLIK", "lar", "koʻplik", None))
for name, c_suf, v_suf in POSS:
    for src in ("OT", "KOʻPLIK", "HARAKAT NOMI"):
        FSA[src].append(("EGALIK", c_suf, "egalik " + name, after_consonant))
        FSA[src].append(("EGALIK", v_suf, "egalik " + name, after_vowel))
for name, suf in CASES[1:]:
    for src in ("OT", "KOʻPLIK", "EGALIK", "HARAKAT NOMI"):
        if suf == "ga":
            FSA[src].append(("KELISHIK", "ga", name, lambda s: not s.endswith(("k", "q"))))
            FSA[src].append(("KELISHIK", "ka", name, lambda s: s.endswith("k")))
            FSA[src].append(("KELISHIK", "qa", name, lambda s: s.endswith("q")))
        else:
            FSA[src].append(("KELISHIK", suf, name, None))
FSA["FEʼL"].append(("INKOR", "ma", "inkor", None))
FSA["FEʼL"].append(("HARAKAT NOMI", "ish", "harakat nomi", after_consonant))
FSA["FEʼL"].append(("HARAKAT NOMI", "sh", "harakat nomi", after_vowel))
for src in ("FEʼL", "INKOR"):
    FSA[src] += [("OXIRI", "", "buyruq mayli, II sh.", None),
                 ("OXIRI", "di", "oʻtgan zamon, III sh.", None), ("OXIRI", "dim", "oʻtgan zamon, I sh.", None),
                 ("OXIRI", "dik", "oʻtgan zamon, I sh. koʻp.", None),
                 ("OXIRI", "adi", "hozirgi-kelasi zamon, III sh.", after_consonant),
                 ("OXIRI", "ydi", "hozirgi-kelasi zamon, III sh.", after_vowel),
                 ("OXIRI", "yapti", "hozirgi zamon, III sh.", None),
                 ("OXIRI", "gan", "sifatdosh", lambda s: not s.endswith(("k", "q"))),
                 ("OXIRI", "kan", "sifatdosh", lambda s: s.endswith("k")),           # ekkan
                 ("OXIRI", "qan", "sifatdosh", lambda s: s.endswith("q"))]           # boqqan
FINAL = {"OT", "KOʻPLIK", "EGALIK", "KELISHIK", "HARAKAT NOMI", "OXIRI"}
START = {"ot": "OT", "feʼl": "FEʼL"}

def walk(rest, state, so_far, path):
    found = [path] if rest == "" and state in FINAL else []
    for nxt, suf, gloss, cond in FSA[state]:
        if rest.startswith(suf) and (cond is None or cond(so_far)):
            found += walk(rest[len(suf):], nxt, so_far + suf, path + [(suf or "Ø", gloss)])
    return found

def stem_variants(root):
    # oʻzakning yuzaki shakllari: «teragi» uchun «terag», «ogʻzi» uchun «ogʻz»
    alt = add_possessive(root, "", "")
    return [(root, False)] + ([(alt, True)] if alt != root else [])

def analyze(word):
    word, res = word.lower(), []
    for root, tags in LEX.items():
        for pos in tags.split():
            if pos not in START:
                if word == root:
                    res.append((root, pos, []))
                continue
            for surface, changed in stem_variants(root):
                if not word.startswith(surface):
                    continue
                for path in walk(word[len(surface):], START[pos], surface, []):
                    # oʻzgargan oʻzakdan keyin faqat unli bilan boshlanuvchi egalik keladi
                    if changed and not (path and path[0][1].startswith("egalik") and path[0][0].startswith("i")):
                        continue
                    res.append((root, pos, path))
    total = sum(FREQ.get((r, p), 1) for r, p, _ in res)
    return sorted([(r, p, path, FREQ.get((r, p), 1) / total) for r, p, path in res], key=lambda x: -x[3])

def show(word):
    res = analyze(word)
    print(word, "" if res else "→ avtomat rad etdi")
    for root, pos, path, prob in res:
        parts = " + ".join([root] + [s for s, _ in path])
        print(f"   {prob:.2f}  {parts:<26} [{pos}] {', '.join(g for _, g in path)}")

for w in ["yoz", "yozda", "yozdi", "ot", "otdan", "olma", "olmadi", "oʻqishga", "kitoblarimizdan",
          "qishlogʻimizga", "teragi", "shahrimizdan", "kitobdalar", "kitobimlar"]:
    show(w)

Avtomatni chizamiz: har bir ramka holat, uning ichida shu holatga olib keladigan qoʻshimchalar. Rad etilgan «kitobdalar» va «kitobimlar» soʻzlarini kuzating: KELISHIK va EGALIK holatlaridan KOʻPLIK ga qaytadigan oʻtish yoʻq, shuning uchun notoʻgʻri tartib qabul qilinmaydi.

In [ ]:
import matplotlib.pyplot as plt
import textwrap

def draw_fsa(edges, pos, final, title):
    # ramka — holat, ichida shu holatga olib keladigan qoʻshimchalar
    incoming = {}
    for (a, b), sufs in edges.items():
        incoming.setdefault(b, set()).update(s or "Ø" for s in sufs)
    fig, ax = plt.subplots(figsize=(11, 4.6))
    boxes = {}
    for name, (x, y) in pos.items():
        label = name
        if name in incoming:
            label += "\n" + textwrap.fill(", ".join(sorted(incoming[name], key=lambda s: (len(s), s))), 22)
        boxes[name] = ax.text(x, y, label, ha="center", va="center", fontsize=8, zorder=3,
                              bbox=dict(boxstyle="round,pad=0.5", fc="#DCE6FA", ec="#2B55C9",
                                        lw=2.5 if name in final else 1))
    fig.canvas.draw()
    for (a, b) in edges:
        (x1, y1), (x2, y2) = pos[a], pos[b]
        rad = (-1 if y1 > 1 else 1) / abs(x2 - x1) if y1 == y2 and abs(x2 - x1) > 2.5 else 0.0
        ax.annotate("", xy=(0.5, 0.5), xycoords=boxes[b], xytext=(0.5, 0.5), textcoords=boxes[a],
                    arrowprops=dict(arrowstyle="-|>", color="#555555", lw=1.1, mutation_scale=14,
                                    patchA=boxes[a].get_bbox_patch(), patchB=boxes[b].get_bbox_patch(),
                                    connectionstyle=f"arc3,rad={rad}"))
    xs, ys = [p[0] for p in pos.values()], [p[1] for p in pos.values()]
    ax.set_xlim(min(xs) - 0.9, max(xs) + 0.9)
    ax.set_ylim(min(ys) - 1.2, max(ys) + 1.2)
    ax.axis("off")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
UZ_EDGES = {}
for src, edges in FSA.items():
    for nxt, suf, _, _ in edges:
        UZ_EDGES.setdefault((src, nxt), []).append(suf)
draw_fsa(UZ_EDGES, {"OT": (0, 2), "KOʻPLIK": (2, 2), "EGALIK": (4, 2), "KELISHIK": (6, 2),
                    "FEʼL": (0, -1), "INKOR": (2, -1), "HARAKAT NOMI": (3, 0.5), "OXIRI": (6, -1)},
         final=FINAL, title="Oʻzbek ot va feʼl shakllari uchun chekli avtomat (qalin ramka: yakuniy holat)")

## 4. Stemming yoki lemmatizatsiya?

Kichik matnni olib, har bir soʻz uchun stemmer natijasini (2-boʻlim) va lemmani (3-boʻlimdagi tahlilchining eng ehtimolli tahlili) topamiz. Lugʻatda yoʻq soʻz uchun lemma soʻzning oʻzi deb olinadi va alohida koʻrsatiladi.

In [ ]:
text = """
Yozda bolalar qishloqqa keldi. Qishlogʻimizda baland teraklar bor. Bolalar bogʻda olma terdi.
Ukam olmani olmadi. Men kitobimni ukamga berdim. Ukam kitobni oʻqiyapti. Otam bogʻda ot boqadi.
Biz shaharga qaytdik. Shahrimizda uylar katta. Bolaning ogʻzida olma bor. Yozda kunlar uzun boʻladi.
Bolalar teraklarning soyasida oʻynadi.
"""
words = re.findall(r"[a-zʻʼ]+", text.lower())

In [ ]:
types = sorted(set(words))
rows = []
for w in types:
    parses = analyze(w)
    lemma = parses[0][0] if parses else w
    rows.append((w, stem_uz(w), lemma, len(parses)))
df = pd.DataFrame(rows, columns=["soʻz", "stemmer", "lemma", "tahlillar"])
print(f"tokenlar {len(words)}, tiplar {len(types)}, stemlar {df['stemmer'].nunique()}, lemmalar {df['lemma'].nunique()}")
print()
print("Stemmer va lemmatizator farq qilgan soʻzlar:")
print(df[df["stemmer"] != df["lemma"]].to_string(index=False))
print()
print("Bir nechta tahlilga ega soʻzlar:", ", ".join(df[df["tahlillar"] > 1]["soʻz"]))
print("Lugʻatda yoʻq (lemma = soʻzning oʻzi):", ", ".join(df[df["tahlillar"] == 0]["soʻz"]) or "yoʻq")

counts = [len(types), df["stemmer"].nunique(), df["lemma"].nunique()]
plt.figure(figsize=(5, 3))
plt.bar(["tiplar", "stemlar", "lemmalar"], counts, color=["#888888", "#2B55C9", "#17907E"])
for i, c in enumerate(counts):
    plt.text(i, c + 0.5, str(c), ha="center")
plt.title("Matndagi turli birliklar soni")
plt.ylim(0, max(counts) * 1.2)
plt.tight_layout()
plt.show()

Tahlilchi ham kontekstni koʻrmaydi. «Bolalar bogʻda olma terdi» gapida «olma» ot (meva), lekin tahlilchi uni alohida koʻradi va oʻyinchoq chastotalar boʻyicha feʼlni tanlaydi (ol + ma, «olmagin»): jadvaldagi «olma → ol» xato. «Ot» esa hayvon ham, «otmoq» feʼli ham boʻlishi mumkin. Bunday koʻp maʼnolilikni gapdagi qoʻshni soʻzlar hal qiladi (soʻz turkumlarini belgilash, 8-mavzu).

### Haqiqiy vosita

Oʻzbek tili uchun ochiq vositalardan biri `UzMorphAnalyser` (U. Salaev, PyPI, taxminan 40 KB): stemmer, lemmatizator va morfologik tahlil, qoʻshimchalar jadvaliga asoslangan. Quyidagi katak faqat Colabda ishlaydi (oʻrnatish uchun internet kerak). Uning natijalarini bizning tahlilchi bilan solishtiring. Bizning sinovimizda (0.0.22-versiya) u «teragi» → «terag», «qishlogʻimizga» → «qishlogʻ», «ogʻzida» → «ogʻz» qaytardi: tovush almashinuvi tiklanmadi. Lekin u lugʻatsiz ishlaydi: 1 400 dan ortiq qoʻshimcha qoidasi va kichik istisnolar roʻyxati bilan istalgan soʻzni tahlil qilishga harakat qiladi. Bizning tahlilchi esa faqat 30 ta oʻzakni biladi va ularni oldindan tanlaganmiz. Ikkala holat ham oʻzbek tili uchun ochiq vositalar hali rivojlanish bosqichida ekanini koʻrsatadi.

In [ ]:
!pip install -q UzMorphAnalyser
try:
    from UzMorphAnalyser import UzMorphAnalyser
    uzm = UzMorphAnalyser()
    for w in ["kitoblarimizdan", "qishlogʻimizga", "teragi", "ogʻzida", "olmadi", "olma", "oʻynadi"]:
        ours = analyze(w)
        print(f"{w:<16} UzMorphAnalyser: {uzm.lemmatize(w):<10} bizning tahlilchi: {ours[0][0] if ours else '—'}")
except Exception as e:
    print("UzMorphAnalyser ishga tushmadi (Colabda internet kerak):", e)

## 5. Topshiriqlar

1. `LEX` ga «yurak» (ot) va «oʻrtoq» (ot) oʻzaklarini qoʻshing. `noun_form` bilan ularning egalik × kelishik jadvalini chiqaring va `analyze` «yuragimizda», «oʻrtogʻingizga», «yurakka» soʻzlarini toʻgʻri tahlil qilishini tekshiring.
2. 2-boʻlimdagi stemmerga istisnolar roʻyxatini (`EXCEPTIONS = {"odam", "kun", "pedagog", ...}`) qoʻshing: soʻz istisno boʻlsa yoki kesishdan keyin istisno oʻzak qolsa, kesishni toʻxtating. Jadvaldagi nechta xato tuzaldi?
3. Avtomatga ravishdosh qoʻshimchasini qoʻshing: undoshdan keyin -ib, unlidan keyin -b (olib, oʻqib, oʻynab). «olib», «oʻqib», «oʻynab» soʻzlarini tahlil qiling.
4. 4-boʻlimdagi matnning har bir soʻzi uchun `analyze` tahlillari sonini sanang. Matndagi soʻzlarning (tokenlarning) qancha ulushi koʻp maʼnoli? Eng koʻp tahlilga ega soʻzni toping.
5. -dagi qoʻshimchasini (oʻrin-payt + belgi: «qishlogʻimizdagi») avtomatga qoʻshing. Uning holati qayerda turishi kerak va undan keyin qaysi qoʻshimchalar kelishi mumkin («qishlogʻimizdagilar»)?

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing